In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()
#dataset

#with open("dataset.json", "w") as f:
with open("dataset1.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [10]:
def run_prompt(test_case):
    """Merge the prompt and test case input. then returns the result """
    prompt = f"""Please solve the following task: {test_case['task']}"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [15]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [20]:
def run_test_case(test_case):
    """calls run_prumpt, then grades the results """
    
    output = run_prompt(test_case)
    
    # TODO - grading 
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output" : output,
        "test_case" : test_case,
        "score": score,
        "reasoning": reasoning
    }

In [25]:
from statistics import mean
def run_eval(dataset):
    """Loads the dateset and calls run_test_case with each case """
    
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score:.2f}")
    
    return results

In [26]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 7.33


In [27]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudFormation Template Parser\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport json\nfrom typing import List, Dict, Any\n\ndef extract_resource_logical_ids(template: str) -> List[str]:\n    \"\"\"\n    Parse an AWS CloudFormation template (JSON string) and extract all resource logical IDs.\n    \n    Args:\n        template: A JSON string representing a CloudFormation template\n        \n    Returns:\n        A list of resource logical IDs\n        \n    Raises:\n        json.JSONDecodeError: If the template is not valid JSON\n        ValueError: If the template doesn't have a Resources section\n    \"\"\"\n    try:\n        template_dict = json.loads(template)\n    except json.JSONDecodeError as e:\n        raise json.JSONDecodeError(f\"Invalid JSON template: {e.msg}\", e.doc, e.pos)\n    \n    if \"Resources\" not in template_dict:\n        raise ValueError(\"Template does not contain a 'Resources' section\")\n    \n    res